In [0]:
# Credentials needed for mounting / direct access
SCOPE       = "default2"
STORAGE     = "dlspl21databricks"
CONTAINER   = "janvander0912"
MOUNT_POINT = f"/mnt/{CONTAINER}_legacy"

In [0]:

# Retrieving credentials from Secret Scope
client_id     = dbutils.secrets.get(SCOPE, "sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(SCOPE, "sp-databricks-adls-appkey")
tenant_id     = dbutils.secrets.get(SCOPE, "tenant-id")

In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

# Classic mount attempt
try:
    dbutils.fs.mount(
        source = f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net/",
        mount_point = MOUNT_POINT,
        extra_configs = configs
    )
    print(f"Successfully mounted at: {MOUNT_POINT}")
except Exception as e:
    pass
    print(f"Mounting error: {e}")

# Direct session setup
account_fqdn = f"{STORAGE}.dfs.core.windows.net"

for key, value in configs.items():
    spark.conf.set(f"{key}.{account_fqdn}", value)

# Listing content using direct path
direct_path = f"abfss://{CONTAINER}@{account_fqdn}/"

display(dbutils.fs.ls(direct_path))